# 🌌 IA-A (EMISOR): HuggingFace Libre + PMTP

Este cuaderno descarga un modelo libre, realiza una inferencia y empaca la respuesta junto a un video/audio/texto en un solo Latent Tensor (.pmtp).

In [ ]:
!pip install -q transformers torch jax jaxlib

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
BUS_DIR = '/content/drive/MyDrive/POLYDIM_BUS/DEMO_PMTP/bus'
os.makedirs(BUS_DIR, exist_ok=True)

### Inferencia con Hugging Face (Modelo Libre)

In [ ]:
from transformers import pipeline
import time

print('Descargando modelo ligero libre (GPT2)...')
t0 = time.time()
generator = pipeline('text-generation', model='gpt2')
respuesta_hf = generator('The future of AI communication is', max_length=30, num_return_sequences=1)[0]['generated_text']
print(f'\n[Inferencia HF completada en {time.time()-t0:.2f}s]\nGenerado: {respuesta_hf}')

### Empaquetado PMTP (Multimedia + HF Inference)

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import struct

# Simulamos leer multimedia cruda (ej. Video/Audio/Texto)
# En este demo, la 'multimedia' y la 'respuesta_hf' se empacan juntas.
data_payload = respuesta_hf.encode('utf-8')

file_size = len(data_payload)
header = bytearray(264)
name = b'huggingface_inference.txt'
header[:len(name)] = name
header[256:264] = struct.pack('<Q', file_size)

total_bytes = header + data_payload
N = int(np.ceil(len(total_bytes) / 1024))
padded_b = bytearray(N * 1024)
padded_b[:len(total_bytes)] = total_bytes

arr_f32 = np.frombuffer(padded_b, dtype=np.int8).astype(np.float32).reshape(N, 1024)
tensor = jnp.array(arr_f32)
jax.block_until_ready(tensor)

pmtp_path = os.path.join(BUS_DIR, 'pmtp_hf_transfer.pmtp')
fp = np.memmap(pmtp_path, dtype=np.float32, mode='w+', shape=(N, 1024))
fp[:] = np.asarray(tensor)[:]
fp.flush()
del fp

print('✅ Tensor PMTP Multimodal y Libre inyectado en el BUS.')